In [2]:
"""
Retrieve data from the Finaledb database and save it to a CSV file.
"""
import requests
import pandas as pd
from glob import glob

# List of IDs to query
ids = []
for i in range(87786, 88325, 1):  # Not inclusive, so add 1 to upper range
    ids.append('EE' + str(i))

# Output CSV file name
output_file = 'cris.csv'

# Check if any of the IDs are already in some output file substring in the cris folder
cris_folder_files = glob('cris/*.bam')
existing_ids = [f.split('/')[-1].split('.')[0] for f in cris_folder_files]
ids = [id_ for id_ in ids if id_ not in existing_ids]
print(f"Number of IDs to process: {len(ids)}")
print(f"Number of IDs already in cris folder: {len(existing_ids)}")

# Initialize a list to store all result dicts
info_table = pd.DataFrame()

for seq_id in ids:
    url = f'http://finaledb.research.cchmc.org/api/v1/seqrun?id={seq_id}'
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            # If data['results'] is a list, we take the first element, else we handle it as a single result
            try:
                if not isinstance(data['results'], list):
                    result = data['results']
                else:
                    result = data['results'][0]
            except:
                print(f"Error processing results for ID {seq_id}: {data['results']}")
                continue
            
            if 'sample' not in result:
                print(f"No sample data for ID {seq_id}")
                continue

            # Convert the result to a DataFrame
            result_dict = {
                'ID': 'EE' + str(result.get('id')),
                'sample_name': result['sample']['name'],
                'assay': result['assay'],
                'readlen': result['seqConfig']['readlen'],
                'instrument': result['seqConfig']['instrument'],
                'seq_layout': result['seqConfig']['seq_layout'],
                'fragNum': result['fragNum']['hg38'],
                'publication_name': result['publication']['citeShort'],
                'publication_doi': result['publication']['identifiers']['doi'],
                'tissue': result['sample']['tissue'],
                'disease': result['sample']['disease'],
            }
            # Append the result DataFrame to the info_table DataFrame
            info_table = pd.concat([info_table, pd.DataFrame([result_dict])], ignore_index=True)

        else:
            print(f"No results found for ID {seq_id}")

    except requests.exceptions.RequestException as e:
        print(f"Request failed for ID {seq_id}: {e}")

info_table.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")


Number of IDs to process: 539
Number of IDs already in cris folder: 0
No sample data for ID EE87808
No sample data for ID EE87930
Data saved to cris.csv
